In [3]:
from collections import defaultdict
import pandas as pd
import json

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 1.3 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.8/341.8 kB 2.2 MB/s eta 0:00:00m eta 0:00:010:00:01


In [38]:
df = pd.read_csv("KoMa-test2.csv")
with open("KoMa-test-meta.json", "r") as f:
    lp_in = json.load(f)

In [18]:
ak_names = [col_name for col_name in df.columns if "Unnamed" not in col_name]
assert len(ak_names) == len(set(ak_names)), "duplicate ak names"
print(ak_names)
df

['ak1', 'ak2']


,Unnamed: 0,ak1,ak2,Unnamed: 3
Lorenzo2,NaN,Nein,Nein,NaN
Test,NaN,Ja,Unter Vorbehalt,NaN
noch jemand,NaN,Ja,Ja,NaN
ein kömischer Name,NaN,Nein,Unter Vorbehalt,NaN


In [39]:
ak_name_to_id_dict = {ak['info']['name']: ak['id'] for ak in lp_in['aks']}
part_name_to_id_dict = {part['info']['name']: part['id'] for part in lp_in['participants']}
print(ak_name_to_id_dict)

{'name': 'Alice'}
participant0
{'name': 'Bob'}
participant1
{'name': 'Claire'}
participant2
{'name': 'David'}
participant3
{'name': 'Emil'}
participant4
{'name': 'Fried'}
participant5
{'name': 'Gerti'}
participant6
{'name': 'Hans'}
participant7
{'AK KoMa': 'ak00', 'AK Website': 'ak01', 'AK BMBF': 'ak02', 'AK Advent': 'ak03', 'AK Minimalstandard': 'ak04', 'AK KdV': 'ak05', 'AK Handzeichen': 'ak06', 'AK Pella': 'ak07', 'AK Grundausstattung': 'ak08', 'AK Onlinewahlen': 'ak09', 'AK CHE': 'ak10', 'AK Taschenrechner': 'ak11', 'ak1': 'ak12', 'ak2': 'ak13'}


In [56]:
def calc_pref_score(entry: str) -> int:
    if entry in ["Nein", "No"]:
        return 0
    elif entry in ["Ja", "Yes"]:
        return 2
    elif entry in ["Unter Vorbehalt", "Under reserve"]:
        return 1
    else:
        raise NotImplementedError
        
def get_part_dict(entries):
    name = entries.name
    if name in part_name_to_id_dict:
        return lp_in['participants'][part_name_to_id_dict[name]]
    i = 0
    while f"participant{i}" in part_name_to_id_dict.values():
        i+=1
    part_id = f"participant{i}"
    part_name_to_id_dict[part_id] = name
    participant_dict = {
        "id": part_id,
        "info": {"name": name},
        "time_constraints": [],
        "room_constraints": [],
        "preferences": [],
    }
    return participant_dict

In [57]:
result_dict = lp_in

index_lst = []
for idx, entries in df.iterrows():
    
    participant_dict = get_part_dict(entries)

    for ak_name in ak_names:
        score = calc_pref_score(entries[ak_name])
        if score != 0:
            participant_dict["preferences"].append(
                {
                    "ak_id": ak_name_to_id_dict[ak_name],
                    "required": False,
                    "preference_score": score,
                }
            )
    if not participant_dict["preferences"]:
        continue
    result_dict["participants"].append(participant_dict)
    index_lst.append(idx)

assert len(index_lst) == len(set(index_lst)), "duplicate participant names"

In [58]:
result_dict

{'rooms': [{'id': 'room0',
   'info': {'name': 'Raum 0.003'},
   'capacity': 3,
   'fulfilled_room_constraints': ['Beamer'],
   'time_constraints': ['room0.003']},
  {'id': 'room1',
   'info': {'name': 'Raum 0.008'},
   'capacity': 6,
   'fulfilled_room_constraints': ['Tafel', 'Beamer'],
   'time_constraints': ['room0.008']},
  {'id': 'room2',
   'info': {'name': 'Zeichensaal'},
   'capacity': 10,
   'fulfilled_room_constraints': ['Tafel'],
   'time_constraints': ['Zeichensaal']}],
 'timeslots': {'info': {'duration': '2 hour blocks'},
  'blocks': [[{'id': 'timeslot0',
     'info': {'start': 'Donnerstag 8:00'},
     'fulfilled_time_constraints': ['Reso',
      'room0.003',
      'room0.008',
      'Zeichensaal',
      'ak02',
      'ak06']},
    {'id': 'timeslot1',
     'info': {'start': 'Donnerstag 10:00'},
     'fulfilled_time_constraints': ['Reso',
      'room0.003',
      'room0.008',
      'Zeichensaal',
      'ak02']}],
   [{'id': 'timeslot2',
     'info': {'start': 'Freitag 14:00

In [59]:
with open("export.json", "w") as f:
    json.dump(result_dict, f, indent=4)